# LatentSync 1.5 口型同步（女主段 · T4 安全版）

为什么是 1.5 不是 1.6：官方 README 写明推理显存 1.5 需 8GB、1.6 需 18GB；免费 Colab T4 只有约 15GB，1.6 装不下，1.5 才跑得动。仓库默认的 inference.sh 现在指向 1.6（stage2_512.yaml），所以这里不用它，直接调 scripts.inference 并指定 1.5 的权重与 stage2.yaml。

先把运行时改成 T4 GPU：菜单 运行时 → 更改运行时类型 → T4 GPU → 保存。然后从上往下逐格运行，不要空等；跑完立刻断开运行时，别让会话空转烧额度。出现付费提示立刻停手。

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone -q https://github.com/bytedance/LatentSync.git
%cd LatentSync
!pip -q install -r requirements.txt

In [ ]:
# 只下 1.5 的权重（setup_env.sh 默认下的是 1.6，T4 跑不动，别用它）
!mkdir -p checkpoints/whisper
!wget -q "https://huggingface.co/ByteDance/LatentSync-1.5/resolve/main/latentsync_unet.pt" -O checkpoints/latentsync_unet.pt
!wget -q "https://huggingface.co/ByteDance/LatentSync-1.5/resolve/main/whisper/tiny.pt" -O checkpoints/whisper/tiny.pt
!ls -lh checkpoints/latentsync_unet.pt checkpoints/whisper/tiny.pt

In [ ]:
!wget -q "https://253631672.github.io/macro-bridge-7f3k2/lipsync/girl_seg.mp4" -O /content/girl_seg.mp4
!wget -q "https://253631672.github.io/macro-bridge-7f3k2/lipsync/girl.wav" -O /content/girl.wav
!ls -lh /content/girl_seg.mp4 /content/girl.wav

In [ ]:
!python -m scripts.inference --unet_config_path "configs/unet/stage2.yaml" --inference_ckpt_path "checkpoints/latentsync_unet.pt" --inference_steps 20 --guidance_scale 1.0 --video_path "/content/girl_seg.mp4" --audio_path "/content/girl.wav" --video_out_path "/content/girl_latentsync.mp4"
!ls -lh /content/girl_latentsync.mp4

In [ ]:
from google.colab import files
files.download('/content/girl_latentsync.mp4')